In [1]:

import pickle, numpy as np
db = pickle.load(open('/n/holylabs/LABS/alvarez_lab/Lab/VVS_Accentuation/Encoding_models/red_20250428-20250430/posthoc_model_predict/accentuated_images_fourier_spectra/accentuated_images_fourier_spectra_db.pkl', 'rb'))
# check spectrum length and a sample image
spec = db['spectrum'].iloc[0]
print("spectrum length:", len(spec))
# check an actual image size
sample_fp = db['filepath'].iloc[0]
print("sample filepath:", sample_fp)
from PIL import Image
import os
if os.path.exists(sample_fp):
    img = Image.open(sample_fp)
    print("image size:", img.size)


spectrum length: 158
sample filepath: /n/holylabs/LABS/alvarez_lab/Everyone/Accentuate_VVS/accentuation_outputs/02-05-2025_red_20250428-20250430_AlexNet_training_seed_01_accentuation/AlexNet_training_seed_01_RidgeCV_unit_0_img_0_level_-0.2282366166114813_score_-0.22128941118717194.png
image size: (1024, 1024)


In [2]:

import pickle
# peek at one PCA pkl to understand structure
d = pickle.load(open('/n/holylabs/LABS/alvarez_lab/Lab/VVS_Accentuation/Encoding_models/red_20250428-20250430/posthoc_model_predict_PCA_popul_unit/posthoc_prediction_PCA_pop_unit_red_20250428-20250430_unit0_resnet50.pkl', 'rb'))
print(type(d))
if isinstance(d, dict):
    for k, v in d.items():
        print(k, type(v), getattr(v, 'shape', ''))
elif hasattr(d, 'columns'):
    print(d.columns.tolist())
    print(d.shape)
    print(d.head(2))


<class 'dict'>
config <class 'dict'> 
df <class 'pandas.core.frame.DataFrame'> (5500, 8)
PCA_resp <class 'torch.Tensor'> torch.Size([5500, 750])
population_resp <class 'torch.Tensor'> torch.Size([5500, 64])
target_unit_resp <class 'torch.Tensor'> torch.Size([5500, 1])
readout_vec <class 'torch.Tensor'> torch.Size([750])
readout_bias <class 'torch.Tensor'> torch.Size([])
cosine_sims <class 'numpy.ndarray'> (5500,)
PCA_norm <class 'torch.Tensor'> torch.Size([5500])


In [3]:

import pickle, numpy as np, pandas as pd

base = '/n/holylabs/LABS/alvarez_lab/Lab/VVS_Accentuation/Encoding_models/red_20250428-20250430/posthoc_model_predict_PCA_popul_unit/'

# Load all 4 relevant pkls
pkls = {}
for accentuator in ['resnet50', 'resnet50_robust']:
    for reviewer in ['resnet50', 'resnet50_robust']:
        key = f"acc={accentuator}_rev={reviewer}"
        # check if cross-model pkl exists
        import os, glob
        pattern = f"{base}*unit15_{reviewer}.pkl"
        files = glob.glob(pattern)
        print(f"{key}: pattern={pattern}")
        print(f"  found: {[os.path.basename(f) for f in files]}")


acc=resnet50_rev=resnet50: pattern=/n/holylabs/LABS/alvarez_lab/Lab/VVS_Accentuation/Encoding_models/red_20250428-20250430/posthoc_model_predict_PCA_popul_unit/*unit15_resnet50.pkl
  found: ['posthoc_prediction_NSDencimg_PCA_pop_unit_red_20250428-20250430_unit15_resnet50.pkl', 'posthoc_prediction_PCA_pop_unit_red_20250428-20250430_unit15_resnet50.pkl']
acc=resnet50_rev=resnet50_robust: pattern=/n/holylabs/LABS/alvarez_lab/Lab/VVS_Accentuation/Encoding_models/red_20250428-20250430/posthoc_model_predict_PCA_popul_unit/*unit15_resnet50_robust.pkl
  found: ['posthoc_prediction_NSDencimg_PCA_pop_unit_red_20250428-20250430_unit15_resnet50_robust.pkl', 'posthoc_prediction_PCA_pop_unit_red_20250428-20250430_unit15_resnet50_robust.pkl']
acc=resnet50_robust_rev=resnet50: pattern=/n/holylabs/LABS/alvarez_lab/Lab/VVS_Accentuation/Encoding_models/red_20250428-20250430/posthoc_model_predict_PCA_popul_unit/*unit15_resnet50.pkl
  found: ['posthoc_prediction_NSDencimg_PCA_pop_unit_red_20250428-20250430

In [4]:

# Each reviewer pkl contains ALL accentuated images (all models), not just same-model
# Let's verify by checking the df inside the pkl
d = pickle.load(open(f"{base}posthoc_prediction_PCA_pop_unit_red_20250428-20250430_unit15_resnet50_robust.pkl", 'rb'))

print("Keys:", list(d.keys()))
print("PCA_resp shape:", d['PCA_resp'].shape)
print("df shape:", d['df'].shape)
print("df columns:", d['df'].columns.tolist())
print("\ndf model_name value counts:")
print(d['df']['model_name'].value_counts())
print("\ndf sample:")
print(d['df'][['model_name','unit_id','img_id','level']].head(5))


Keys: ['config', 'df', 'PCA_resp', 'population_resp', 'target_unit_resp', 'readout_vec', 'readout_bias', 'cosine_sims', 'PCA_norm']
PCA_resp shape: torch.Size([5500, 750])
df shape: (5500, 8)
df columns: ['model_name', 'unit_id', 'img_id', 'level', 'score', 'filepath', 'cosine_similarity', 'PCA_norm']

df model_name value counts:
model_name
AlexNet_training_seed_01    550
clipag_vitb32               550
dinov2_vitb14_reg           550
radio_v2.5-b                550
regnety_640                 550
resnet50                    550
resnet50_clip               550
resnet50_dino               550
resnet50_robust             550
siglip2_vitb16              550
Name: count, dtype: int64

df sample:
                 model_name  unit_id  img_id     level
0  AlexNet_training_seed_01        0       0 -0.228237
1  AlexNet_training_seed_01        0       0 -1.037346
2  AlexNet_training_seed_01        0       0 -1.846455
3  AlexNet_training_seed_01        0       0  0.580873
4  AlexNet_training_seed

In [5]:

# NSD pkl
d_nsd = pickle.load(open(f"{base}posthoc_prediction_NSDencimg_PCA_pop_unit_red_20250428-20250430_unit15_resnet50_robust.pkl", 'rb'))
print("NSD PCA_resp shape:", d_nsd['PCA_resp'].shape)
print("NSD df columns:", d_nsd['df'].columns.tolist())
print("NSD df head:")
print(d_nsd['df'][['stimulus_name','is_nsd','is_train','is_test']].head(3))
print("\nNSD target_unit_resp shape:", d_nsd['target_unit_resp'].shape)
print("NSD target_unit_resp range:", d_nsd['target_unit_resp'].min().item(), d_nsd['target_unit_resp'].max().item())


NSD PCA_resp shape: torch.Size([969, 750])
NSD df columns: ['stimulus_name', 'is_nsd', 'is_floc', 'is_OO', 'is_train', 'is_normalizer', 'is_test', 'image_fps']
NSD df head:
       stimulus_name  is_nsd  is_train is_test
0  BigAnimate_20.png   False      True   False
1  BigAnimate_60.png   False      True   False
2        FFM_022.png   False      True   False

NSD target_unit_resp shape: torch.Size([969, 1])
NSD target_unit_resp range: -0.8007699847221375 3.6123645305633545


In [6]:

import matplotlib
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, '/n/home12/binxuwang/Github/Closed-loop-visual-insilico')
from circuit_toolkit.plot_utils import saveallforms
import torch

SUBJECT = 'red_20250428-20250430'
UNIT = 15
ACCENTUATORS = ['resnet50', 'resnet50_robust']
REVIEWERS     = ['resnet50', 'resnet50_robust']
LABELS = {'resnet50': 'ResNet50', 'resnet50_robust': 'ResNet50-Robust'}
COLORS = {'resnet50': '#4878CF', 'resnet50_robust': '#FF8C00'}

def load_reviewer_data(subject, unit, reviewer):
    p = f"{base}posthoc_prediction_PCA_pop_unit_{subject}_unit{unit}_{reviewer}.pkl"
    return pickle.load(open(p, 'rb'))

def load_nsd_data(subject, unit, reviewer):
    p = f"{base}posthoc_prediction_NSDencimg_PCA_pop_unit_{subject}_unit{unit}_{reviewer}.pkl"
    return pickle.load(open(p, 'rb'))

# Test: get PC1/PC2 for resnet50 accentuated in resnet50_robust PCA
d_rev = load_reviewer_data(SUBJECT, UNIT, 'resnet50_robust')
d_nsd = load_nsd_data(SUBJECT, UNIT, 'resnet50_robust')

pca_all = d_rev['PCA_resp'].numpy()  # (5500, 750)
df_all  = d_rev['df']
pca_nsd = d_nsd['PCA_resp'].numpy()  # (969, 750)

# Filter to unit 15, resnet50 accentuated
mask = (df_all['model_name'] == 'resnet50') & (df_all['unit_id'] == UNIT)
pca_acc = pca_all[mask.values]
df_acc  = df_all[mask]
levels  = df_acc['level'].values

print(f"acc shape: {pca_acc.shape}, levels range: {levels.min():.2f} to {levels.max():.2f}")
print(f"nsd shape: {pca_nsd.shape}")
print(f"PC1 range acc: {pca_acc[:,0].min():.2f} to {pca_acc[:,0].max():.2f}")
print(f"PC1 range nsd: {pca_nsd[:,0].min():.2f} to {pca_nsd[:,0].max():.2f}")


acc shape: (110, 750), levels range: -1.71 to 4.90
nsd shape: (969, 750)
PC1 range acc: -6.21 to 40.16
PC1 range nsd: -111.24 to 49.00


In [7]:

# Build the 2x2 figure: rows=reviewer, cols=accentuator
# Each panel: NSD cloud (gray) + accentuated trajectory colored by level + seed images as separate traces

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

for ri, reviewer in enumerate(REVIEWERS):
    d_rev = load_reviewer_data(SUBJECT, UNIT, reviewer)
    d_nsd = load_nsd_data(SUBJECT, UNIT, reviewer)
    pca_all = d_rev['PCA_resp'].numpy()
    df_all  = d_rev['df']
    pca_nsd = d_nsd['PCA_resp'].numpy()

    for ci, accentuator in enumerate(ACCENTUATORS):
        ax = axes[ri, ci]
        
        # NSD reference cloud
        ax.scatter(pca_nsd[:, 0], pca_nsd[:, 1], c='lightgray', s=8, alpha=0.4, 
                   label='NSD (~1k)', zorder=1, rasterized=True)
        
        # Accentuated images: filter to this accentuator, unit 15
        mask = (df_all['model_name'] == accentuator) & (df_all['unit_id'] == UNIT)
        pca_acc = pca_all[mask.values]
        df_acc  = df_all[mask]
        
        # Group by img_id, plot each seed as a trajectory colored by level
        cmap = plt.cm.RdBu_r
        levels_all = df_acc['level'].values
        lv_min, lv_max = levels_all.min(), levels_all.max()
        
        for img_id, grp in df_acc.groupby('img_id'):
            idx = grp.index - df_acc.index[0]  # relative index into pca_acc
            # Use positional indexing
            grp_pos = df_acc.index.get_indexer(grp.index)
            pca_grp = pca_acc[grp_pos]
            lvs = grp['level'].values
            order = np.argsort(lvs)
            pca_sorted = pca_grp[order]
            lvs_sorted  = lvs[order]
            
            # Color-coded scatter
            sc = ax.scatter(pca_sorted[:, 0], pca_sorted[:, 1], 
                           c=lvs_sorted, cmap=cmap, vmin=lv_min, vmax=lv_max,
                           s=30, alpha=0.8, zorder=3)
            # Connect with thin line
            ax.plot(pca_sorted[:, 0], pca_sorted[:, 1], 
                    color='gray', lw=0.5, alpha=0.3, zorder=2)
        
        acc_c  = COLORS[accentuator]
        rev_c  = COLORS[reviewer]
        ax.set_xlabel(f'PC1 ({LABELS[reviewer]} space)', fontsize=9)
        ax.set_ylabel(f'PC2 ({LABELS[reviewer]} space)', fontsize=9)
        ax.set_title(f'Accentuator: {LABELS[accentuator]}\nReviewer PCA: {LABELS[reviewer]}', 
                     fontsize=10, color=acc_c)
        ax.tick_params(labelsize=8)

# Add colorbar
fig.subplots_adjust(right=0.88, hspace=0.35, wspace=0.3)
cbar_ax = fig.add_axes([0.90, 0.15, 0.02, 0.70])
sm = plt.cm.ScalarMappable(cmap=plt.cm.RdBu_r, norm=plt.Normalize(lv_min, lv_max))
fig.colorbar(sm, cax=cbar_ax, label='Accentuation level')

fig.suptitle(f'{SUBJECT}  unit {UNIT}\nCross-model PCA: accentuated image activations\n(gray = NSD ~1k reference)', 
             fontsize=12, fontweight='bold')

FIGDIR = '/n/home12/binxuwang/Github/Closed-loop-visual-insilico/figures/red_20250428-20250430_resnet50_vs_resnet50_robust_freq_comparison'
saveallforms(FIGDIR, f'{SUBJECT}_U{UNIT}_crossmodel_PCA_resnet50_vs_robust', figh=fig)
plt.show()
print("Saved")


Saved


[05/13/26 20:03:11] INFO     maxp pruned                        __init__.py:3333
                    INFO     cmap pruned                        __init__.py:3333
                    INFO     kern dropped                       __init__.py:3317
                    INFO     post pruned                        __init__.py:3333
                    INFO     FFTM dropped                       __init__.py:3317
                    INFO     GPOS pruned                        __init__.py:3333
                    INFO     GSUB pruned                        __init__.py:3333
                    INFO     glyf pruned                        __init__.py:3333
                    INFO     Added gid0 to subset               __init__.py:3373
                    INFO     Added first four glyphs to subset  __init__.py:3381
                    INFO     Closing glyph list over 'GSUB': 42 __init__.py:3399
                             glyphs before                                      
                    INFO    

In [8]:

# Plot 2: PC activation profile colored by level
# For each level, show mean activation across all img_ids at each PC index

fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharey=False)

for ri, reviewer in enumerate(REVIEWERS):
    d_rev = load_reviewer_data(SUBJECT, UNIT, reviewer)
    pca_all = d_rev['PCA_resp'].numpy()
    df_all  = d_rev['df']

    for ci, accentuator in enumerate(ACCENTUATORS):
        ax = axes[ri, ci]

        mask = (df_all['model_name'] == accentuator) & (df_all['unit_id'] == UNIT)
        pca_acc = pca_all[mask.values]   # (110, 750)
        df_acc  = df_all[mask]
        levels_all = df_acc['level'].values

        lv_min, lv_max = levels_all.min(), levels_all.max()
        cmap = plt.cm.RdBu_r

        # Group by level (each level has 10 img_ids), plot mean ± sem per PC
        n_pcs = 80  # show first 80 PCs
        pcs = np.arange(n_pcs)
        for lv_val in sorted(df_acc['level'].unique()):
            lv_mask = df_acc['level'].values == lv_val
            mean_act = pca_acc[lv_mask, :n_pcs].mean(axis=0)
            lv_norm = (lv_val - lv_min) / (lv_max - lv_min)
            c = cmap(lv_norm)
            ax.plot(pcs, mean_act, color=c, lw=0.9, alpha=0.55)

        ax.axhline(0, color='k', lw=0.6, ls='--', alpha=0.4)
        ax.set_xlabel('PC index', fontsize=9)
        ax.set_ylabel('Mean activation', fontsize=9)
        ax.set_title(f'Accentuator: {LABELS[accentuator]}\nReviewer: {LABELS[reviewer]}',
                     fontsize=10, color=COLORS[accentuator])
        ax.tick_params(labelsize=8)
        ax.set_xlim(0, n_pcs-1)

fig.subplots_adjust(right=0.88, hspace=0.42, wspace=0.3)
cbar_ax = fig.add_axes([0.90, 0.15, 0.02, 0.70])
sm = plt.cm.ScalarMappable(cmap=plt.cm.RdBu_r, norm=plt.Normalize(lv_min, lv_max))
fig.colorbar(sm, cax=cbar_ax, label='Accentuation level')
fig.suptitle(f'{SUBJECT}  unit {UNIT} — Mean activation by PC index\n(each line = one level, color = level)',
             fontsize=12, fontweight='bold')
saveallforms(FIGDIR, f'{SUBJECT}_U{UNIT}_crossmodel_PC_profile_resnet50_vs_robust', figh=fig)
plt.show()
print("Saved")


Saved


[05/13/26 20:05:37] INFO     maxp pruned                        __init__.py:3333
                    INFO     cmap pruned                        __init__.py:3333
                    INFO     kern dropped                       __init__.py:3317
                    INFO     post pruned                        __init__.py:3333
                    INFO     FFTM dropped                       __init__.py:3317
                    INFO     GPOS pruned                        __init__.py:3333
                    INFO     GSUB pruned                        __init__.py:3333
                    INFO     glyf pruned                        __init__.py:3333
                    INFO     Added gid0 to subset               __init__.py:3373
                    INFO     Added first four glyphs to subset  __init__.py:3381
                    INFO     Closing glyph list over 'GSUB': 37 __init__.py:3399
                             glyphs before                                      
                    INFO    

In [9]:

# Plot 3: Correlation of each PC with accentuation level
# This shows which PCs encode the accentuation direction most strongly

from scipy.stats import pearsonr

fig, axes = plt.subplots(2, 2, figsize=(14, 7), sharey=False)

for ri, reviewer in enumerate(REVIEWERS):
    d_rev = load_reviewer_data(SUBJECT, UNIT, reviewer)
    pca_all = d_rev['PCA_resp'].numpy()
    df_all  = d_rev['df']

    for ci, accentuator in enumerate(ACCENTUATORS):
        ax = axes[ri, ci]

        mask = (df_all['model_name'] == accentuator) & (df_all['unit_id'] == UNIT)
        pca_acc = pca_all[mask.values]   # (110, 750)
        levels  = df_all[mask]['level'].values

        n_pcs = 100
        corrs = np.array([pearsonr(pca_acc[:, k], levels)[0] for k in range(n_pcs)])

        c = COLORS[accentuator]
        ax.bar(np.arange(n_pcs), corrs, color=c, alpha=0.6, width=0.8)
        ax.axhline(0, color='k', lw=0.7)
        ax.set_xlabel('PC index', fontsize=9)
        ax.set_ylabel('Pearson r with level', fontsize=9)
        ax.set_title(f'Accentuator: {LABELS[accentuator]}\nReviewer: {LABELS[reviewer]}',
                     fontsize=10, color=c)
        ax.set_xlim(-1, n_pcs)
        ax.tick_params(labelsize=8)

fig.suptitle(f'{SUBJECT}  unit {UNIT} — PC correlation with accentuation level\n'
             f'(which PCs encode the accentuation direction?)',
             fontsize=12, fontweight='bold')
fig.tight_layout()
saveallforms(FIGDIR, f'{SUBJECT}_U{UNIT}_crossmodel_PC_level_corr_resnet50_vs_robust', figh=fig)
plt.show()
print("Saved")


Saved


[05/13/26 20:05:47] INFO     maxp pruned                        __init__.py:3333
                    INFO     cmap pruned                        __init__.py:3333
                    INFO     kern dropped                       __init__.py:3317
                    INFO     post pruned                        __init__.py:3333
                    INFO     FFTM dropped                       __init__.py:3317
                    INFO     GPOS pruned                        __init__.py:3333
                    INFO     GSUB pruned                        __init__.py:3333
                    INFO     glyf pruned                        __init__.py:3333
                    INFO     Added gid0 to subset               __init__.py:3373
                    INFO     Added first four glyphs to subset  __init__.py:3381
                    INFO     Closing glyph list over 'GSUB': 34 __init__.py:3399
                             glyphs before                                      
                    INFO    

In [10]:

# Plot 2 revised: all 750 PCs, all raw lines (no mean), colored by level
fig, axes = plt.subplots(2, 2, figsize=(16, 8), sharey=False)

for ri, reviewer in enumerate(REVIEWERS):
    d_rev = load_reviewer_data(SUBJECT, UNIT, reviewer)
    pca_all = d_rev['PCA_resp'].numpy()
    df_all  = d_rev['df']

    for ci, accentuator in enumerate(ACCENTUATORS):
        ax = axes[ri, ci]

        mask = (df_all['model_name'] == accentuator) & (df_all['unit_id'] == UNIT)
        pca_acc = pca_all[mask.values]   # (110, 750)
        df_acc  = df_all[mask]
        levels_all = df_acc['level'].values
        lv_min, lv_max = levels_all.min(), levels_all.max()
        cmap = plt.cm.RdBu_r

        pcs = np.arange(750)
        for i, (lv, row) in enumerate(zip(levels_all, pca_acc)):
            lv_norm = (lv - lv_min) / (lv_max - lv_min)
            ax.plot(pcs, row, color=cmap(lv_norm), lw=0.5, alpha=0.25)

        ax.axhline(0, color='k', lw=0.6, ls='--', alpha=0.4)
        ax.set_xlabel('PC index', fontsize=9)
        ax.set_ylabel('Activation', fontsize=9)
        ax.set_title(f'Accentuator: {LABELS[accentuator]}\nReviewer: {LABELS[reviewer]}',
                     fontsize=10, color=COLORS[accentuator])
        ax.tick_params(labelsize=8)
        ax.set_xlim(0, 749)

fig.subplots_adjust(right=0.88, hspace=0.42, wspace=0.3)
cbar_ax = fig.add_axes([0.90, 0.15, 0.02, 0.70])
sm = plt.cm.ScalarMappable(cmap=plt.cm.RdBu_r, norm=plt.Normalize(lv_min, lv_max))
fig.colorbar(sm, cax=cbar_ax, label='Accentuation level')
fig.suptitle(f'{SUBJECT}  unit {UNIT} — PC activation profile (all 750 dims, all images)\n(each line = one image, color = level)',
             fontsize=12, fontweight='bold')
saveallforms(FIGDIR, f'{SUBJECT}_U{UNIT}_crossmodel_PC_profile_full_resnet50_vs_robust', figh=fig)
plt.show()
print("Saved")


Saved


[05/13/26 20:08:16] INFO     maxp pruned                        __init__.py:3333
                    INFO     cmap pruned                        __init__.py:3333
                    INFO     kern dropped                       __init__.py:3317
                    INFO     post pruned                        __init__.py:3333
                    INFO     FFTM dropped                       __init__.py:3317
                    INFO     GPOS pruned                        __init__.py:3333
                    INFO     GSUB pruned                        __init__.py:3333
                    INFO     glyf pruned                        __init__.py:3333
                    INFO     Added gid0 to subset               __init__.py:3373
                    INFO     Added first four glyphs to subset  __init__.py:3381
                    INFO     Closing glyph list over 'GSUB': 39 __init__.py:3399
                             glyphs before                                      
                    INFO    

In [11]:

# Also redo correlation plot with all 750 PCs
from scipy.stats import pearsonr

fig, axes = plt.subplots(2, 2, figsize=(16, 7), sharey=False)

for ri, reviewer in enumerate(REVIEWERS):
    d_rev = load_reviewer_data(SUBJECT, UNIT, reviewer)
    pca_all = d_rev['PCA_resp'].numpy()
    df_all  = d_rev['df']

    for ci, accentuator in enumerate(ACCENTUATORS):
        ax = axes[ri, ci]

        mask = (df_all['model_name'] == accentuator) & (df_all['unit_id'] == UNIT)
        pca_acc = pca_all[mask.values]
        levels  = df_all[mask]['level'].values

        corrs = np.array([pearsonr(pca_acc[:, k], levels)[0] for k in range(750)])

        c = COLORS[accentuator]
        ax.bar(np.arange(750), corrs, color=c, alpha=0.6, width=1.0)
        ax.axhline(0, color='k', lw=0.7)
        ax.set_xlabel('PC index', fontsize=9)
        ax.set_ylabel('Pearson r with level', fontsize=9)
        ax.set_title(f'Accentuator: {LABELS[accentuator]}\nReviewer: {LABELS[reviewer]}',
                     fontsize=10, color=c)
        ax.set_xlim(-1, 750)
        ax.tick_params(labelsize=8)

fig.suptitle(f'{SUBJECT}  unit {UNIT} — PC correlation with accentuation level (all 750 dims)',
             fontsize=12, fontweight='bold')
fig.tight_layout()
saveallforms(FIGDIR, f'{SUBJECT}_U{UNIT}_crossmodel_PC_level_corr_full_resnet50_vs_robust', figh=fig)
plt.show()
print("Saved")


Saved


[05/13/26 20:08:29] INFO     maxp pruned                        __init__.py:3333
                    INFO     cmap pruned                        __init__.py:3333
                    INFO     kern dropped                       __init__.py:3317
                    INFO     post pruned                        __init__.py:3333
                    INFO     FFTM dropped                       __init__.py:3317
                    INFO     GPOS pruned                        __init__.py:3333
                    INFO     GSUB pruned                        __init__.py:3333
                    INFO     glyf pruned                        __init__.py:3333
                    INFO     Added gid0 to subset               __init__.py:3373
                    INFO     Added first four glyphs to subset  __init__.py:3381
                    INFO     Closing glyph list over 'GSUB': 35 __init__.py:3399
                             glyphs before                                      
                    INFO    

In [12]:

from mcp__ipython_kernel import kernel_save_notebook


ModuleNotFoundError("No module named 'mcp__ipython_kernel'")

---------------------------------------------------------------------------
ModuleNotFoundError                       Traceback (most recent call last)
Cell In[1], line 1
----> 1 from mcp__ipython_kernel import kernel_save_notebook

ModuleNotFoundError: No module named 'mcp__ipython_kernel'
